In [ ]:
#conda activate burnseverity
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import requests
import json

import rioxarray as rxr

import geopandas as gpd

import validation as val



The goal of this notebook is to compare per-pixel dNBR values between our tool and the official BAER assessments for the fire where we have those. The goal is to do the following:
1. Regrid all rasters to 30 m resolution (Landsat)
2. Crop all raster to the respective CalFire boundaries to define a common domain
3. Calculate the per pixel $R^2$ 

In [8]:


fire_event_name = 'GEOLOGY'
job_id = 'd5751455-9f39-4e16-a740-86c48af4033f'
request = requests.get(f"https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/result/analyze_fire_severity/{fire_event_name}/{job_id}")
response_data = request.json()
response_data


{'fire_event_name': 'GEOLOGY',
 'status': 'pending',
 'job_id': 'd5751455-9f39-4e16-a740-86c48af4033f'}

In [ ]:
if response_data.get('status') != 'complete':
    print(f"  Job {fire_event_name} is {response_data.get('status')}")

urls = response_data.get('coarse_severity_cog_urls')
dnbr_url = urls.get('dnbr')
rbr_url = urls.get('rbr')

ref_raster = rxr.open_rasterio("dnbr_url")
plot(ref_raster)

resampled = rxr.open_rasterio("baer/organized/GEOLOGY_2023-06-10/ca3389711605420230610_20230603_20230619_dnbr.tif").rio.reproject_match(ref_30m)